<a href="https://colab.research.google.com/github/SunJ2432/ai-image-compliance/blob/main/notebooks/02_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — DenseNet121 detector (tiny smoke test)

Uses `samples_tiny/manifest.csv`. The model is not validated for detecting AI edits; this notebook only verifies training, checkpointing, and predictions.

In [1]:
%pip install -q timm torch torchvision Pillow pytest

In [5]:
from pathlib import Path
import os
import sys

GITHUB_REPO = "SunJ2432/ai-image-compliance"
repo_dir = Path("/content") / GITHUB_REPO.split("/")[-1]

if not repo_dir.exists():
    # First time: clone repository
    !git clone https://github.com/{GITHUB_REPO}.git {repo_dir}
else:
    # Subsequent runs: fetch latest changes
    !git -C {repo_dir} pull --ff-only

# Verify dataset exists
assert (repo_dir / "samples_tiny/manifest.csv").is_file(), \
    "Check samples_tiny folder and clone path"

# Add src/ to Python's import path
if str(repo_dir / "src") not in sys.path:
    sys.path.insert(0, str(repo_dir / "src"))

print("Repository synchronized:", repo_dir)

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 11 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 4.97 KiB | 1.66 MiB/s, done.
From https://github.com/SunJ2432/ai-image-compliance
   8445c94..6e07dfe  main       -> origin/main
Updating 8445c94..6e07dfe
Fast-forward
 src/imgcompliance/detector/__init__.py |   5 +-
 src/imgcompliance/detector/model.py    | 158 +++++++++++++++++++++++++++++++++
 tests/test_detector.py                 |  29 ++++++
 3 files changed, 191 insertions(+), 1 deletion(-)
 create mode 100644 src/imgcompliance/detector/model.py
 create mode 100644 tests/test_detector.py
Repository synchronized: /content/ai-image-compliance


In [9]:
import importlib
import imgcompliance.detector

importlib.reload(imgcompliance.detector)

from imgcompliance.detector import train, Detector

In [10]:
manifest = repo_dir / "samples_tiny" / "manifest.csv"
checkpoint = Path("/content/detector_head.pt")
detector = train(manifest, epochs=3, batch_size=6, output_path=checkpoint)
print("Checkpoint saved:", checkpoint)

model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

model.safetensors: downloading bytes:           |  0.00B            

epoch 1/3 loss=0.6614
epoch 2/3 loss=0.6131
epoch 3/3 loss=0.5681
Checkpoint saved: /content/detector_head.pt


In [11]:
import csv
with manifest.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
for row in rows:
    path = manifest.parent / row["filename"]
    print(path.name, "ground_truth=", row["label"], "prediction=", detector.predict(path))

real_001.jpg ground_truth= real prediction= {'prob': 0.32856687903404236, 'label': 'real', 'model_version': 'densenet121-frozen-head-v1'}
ai_001.jpg ground_truth= ai_edited prediction= {'prob': 0.5376836061477661, 'label': 'ai', 'model_version': 'densenet121-frozen-head-v1'}
real_002.jpg ground_truth= real prediction= {'prob': 0.48223426938056946, 'label': 'real', 'model_version': 'densenet121-frozen-head-v1'}
ai_002.jpg ground_truth= ai_edited prediction= {'prob': 0.6127488017082214, 'label': 'ai', 'model_version': 'densenet121-frozen-head-v1'}
real_003.jpg ground_truth= real prediction= {'prob': 0.4891802668571472, 'label': 'real', 'model_version': 'densenet121-frozen-head-v1'}
ai_003.jpg ground_truth= ai_edited prediction= {'prob': 0.7271319627761841, 'label': 'ai', 'model_version': 'densenet121-frozen-head-v1'}


In [12]:
# Check checkpoint reload
reloaded = Detector(checkpoint)
print("Reloaded model:", reloaded.predict(manifest.parent / rows[0]["filename"]))

Reloaded model: {'prob': 0.32856687903404236, 'label': 'real', 'model_version': 'densenet121-frozen-head-v1'}
